# Árvores B e B+ em um catálogo de streaming

**Um experimento reproduzível com dados IMDb, índices instrumentados e SQLite**

Este notebook é simultaneamente artigo experimental e programa executável. Todas as decisões, validações, medições e exportações aparecem em células visíveis. Nenhum resultado numérico é preenchido antecipadamente: tabelas, frases analíticas e gráficos são derivados dos CSV gerados nesta execução.


## Resumo do experimento

Investigamos como Árvores B e B+ se comportam ao indexar um catálogo internacional de filmes. Implementações próprias, em memória e sob condições equivalentes recebem as mesmas chaves, referências, ordens de inserção e cargas. Medimos tempo de CPU/parede com relógio monotônico de alta resolução, memória Python rastreada, comparações, divisões e acessos a páginas **lógicas**. Os registros completos permanecem no SQLite; os nós guardam apenas chave e `index_key`.

A referência SQLite contextualiza os resultados em um SGBD real, mas não é tratada como implementação equivalente: seu código é nativo, possui cache, formato de página, otimizador e I/O próprios. O modo rápido verifica o pipeline; o modo completo é destinado à coleta final.


## Problema de pesquisa e hipóteses

**Pergunta:** Como Árvores B e Árvores B+ se comportam, na prática, na indexação e consulta de um grande catálogo de filmes, considerando inserções, buscas pontuais, buscas por intervalo, páginas simuladas e consumo de memória/armazenamento?

Hipóteses, formuladas como expectativas e não como conclusões:

- **H1:** ambas terão crescimento aproximadamente logarítmico em buscas pontuais;
- **H2:** a B+ terá vantagem mais clara em intervalos;
- **H3:** ordens maiores tenderão a reduzir a altura até certo ponto;
- **H4:** inserções ordenadas alterarão splits e ocupação;
- **H5:** implementação e armazenamento podem pesar mais no tempo absoluto que a classe assintótica;
- **H6:** a B pode ser competitiva quando encontra chaves em nós internos.


## Preparação do ambiente

As dependências são instaladas a partir de um arquivo versionado. `pandas`/`numpy` tratam os dados, `kagglehub` usa a distribuição oficial do Kaggle, `psutil` descreve a máquina, Plotly gera figuras e Kaleido permite exportação estática. `sqlite3`, `time`, `tracemalloc`, `gc`, `pathlib`, `statistics`, `json`, `platform` e `subprocess` pertencem à biblioteca padrão.

Não adotamos biblioteca externa de árvores: suas diferenças de armazenamento e API dificultariam isolar a estrutura. As duas implementações próprias têm licença MIT neste projeto, operam no mesmo processo e na RAM, usam a mesma definição de ordem, divisão por mediana/capacidade e instrumentação. Essa decisão melhora a validade interna, embora código Python acadêmico não represente um índice de produção em C com páginas físicas.


In [1]:
from pathlib import Path

candidate = Path.cwd().resolve()
PROJECT_ROOT = candidate if (candidate / "src").exists() else candidate.parent
requirements = PROJECT_ROOT / "requirements.txt"
assert requirements.exists(), f"requirements.txt não encontrado em {PROJECT_ROOT}"
%pip install -q -r "$requirements"



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Configuração única

Cada cenário executa 150 operações em cada uma de duas repetições, totalizando 300 medições por tipo de operação. Os modos `quick` e `full` mantêm essa contagem; `full` habilita a exportação estática. Tamanho completo e imagens estáticas são opções explícitas porque podem exigir muitas horas, memória e um navegador compatível com Kaleido. Todos os caminhos são relativos e a semente fixa é 42.

**Política de execução limpa:** ao executar esta célula, CSVs processados, resultados, gráficos e banco SQLite de execuções anteriores são removidos. O cache original do Kaggle em `data/raw` é preservado. Assim, nenhum resultado antigo pode ser combinado silenciosamente com a execução atual.


In [2]:
import gc
import json
import platform
import sqlite3
import statistics
import subprocess
import sys
import time
import tracemalloc
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import psutil
from IPython.display import Markdown, display

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bplustree import BPlusTree
from src.btree import BTree
from src.benchmark import environment_table, numeric_interpretation
from src.config import ExperimentConfig
from src.data import (
    DATASET_HANDLE, clean_movies, discover_source_files, download_dataset,
    infer_column_mapping, inventory_files, read_dataset, read_initial_sample,
    reproducible_samples,
)
from src.database import connect_database, create_catalog, database_pages
from src.experiment import run_catalog_operation_experiments, save_processed_results
from src.plots import create_figures, export_figures
from src.validation import run_all_validations

RUN_MODE = "quick"  # troque para "full" para o experimento final
ORDENS = [32, 128, 256]
TAMANHOS = [10_000, 100_000, 700_000]
config = ExperimentConfig(run_mode=RUN_MODE, project_root=PROJECT_ROOT)
config.orders = tuple(ORDENS)
config.quick_sizes = tuple(TAMANHOS)
config.full_sizes = tuple(TAMANHOS)
config.include_full_dataset = False
config.export_static_images = RUN_MODE == "full"
config.create_directories(clear_previous_outputs=True)
config


ExperimentConfig(run_mode='quick', random_seed=42, orders=(32, 128, 256), quick_sizes=(10000, 100000, 700000), full_sizes=(10000, 100000, 700000), quick_repetitions=2, full_repetitions=2, quick_queries=150, full_queries=150, include_full_dataset=False, export_static_images=False, project_root=PosixPath('/Users/gabriel_goncalves/Desktop/UFABC/Projeto-Algo/Algoritmos-2-Projeto'))

## Hardware e software

Tempos absolutos só são diretamente comparáveis na mesma máquina e sob condições semelhantes. Registramos sistema, Python, arquitetura, CPU, núcleos, RAM, bibliotecas e horário. O tipo físico de armazenamento não é detectável portavelmente e, por isso, a tabela solicita preenchimento manual em vez de adivinhar.


In [ ]:
environment = environment_table()
environment.to_csv(config.raw_results_dir / "environment.csv", index=False)
display(environment)


,property,value
0,execution_datetime,2026-07-18T12:08:52.249611-03:00
1,operating_system,Darwin 25.2.0
2,python,"3.13.3 (main, Apr 8 2025, 13:54:08) [Clang 17..."
3,architecture,arm64
4,processor,arm
5,logical_cores,8
6,physical_cores,8
7,ram_bytes,8589934592
8,storage_type,SSD/HDD não detectável de forma portátil; pree...
9,pandas,3.0.3


## Dataset e download oficial

O conjunto [IMDb Dataset of 600K International Movies](https://www.kaggle.com/datasets/pavan4kalyan/imdb-dataset-of-600k-international-movies/data) é obtido por `kagglehub`. O Kaggle pode pedir credenciais conforme ambiente e política atual; nesse caso, configure `~/.kaggle/kaggle.json` e execute novamente. Não pressupomos nome nem formato: inventariamos o cache oficial e aceitamos um arquivo delimitado principal ou uma coleção de lotes JSON. Para não duplicar vários gigabytes, `data/raw/kagglehub_location.txt` registra o caminho do cache em vez de copiar seu conteúdo.


In [ ]:
dataset_root = download_dataset(config.raw_data_dir, DATASET_HANDLE)
file_inventory = inventory_files(dataset_root)
display(file_inventory)
source_files = discover_source_files(dataset_root)
print(f"Fontes selecionadas: {len(source_files)}; primeira: {source_files[0].name}")


,path,extension,size_bytes
0,/Users/gabriel_goncalves/.cache/kagglehub/data...,.json,12379006
1,/Users/gabriel_goncalves/.cache/kagglehub/data...,.json,12379006
2,/Users/gabriel_goncalves/.cache/kagglehub/data...,.json,12175316
3,/Users/gabriel_goncalves/.cache/kagglehub/data...,.json,11825152
4,/Users/gabriel_goncalves/.cache/kagglehub/data...,.json,11722318
...,...,...,...
709,/Users/gabriel_goncalves/.cache/kagglehub/data...,.json,3459212
710,/Users/gabriel_goncalves/.cache/kagglehub/data...,.json,3454774
711,/Users/gabriel_goncalves/.cache/kagglehub/data...,.json,3442888
712,/Users/gabriel_goncalves/.cache/kagglehub/data...,.json,3416962


Fontes selecionadas: 713; primeira: movies_batch_1.json


### Descoberta de codificação, separador e esquema

Para arquivo delimitado, testamos codificações comuns e usamos `csv.Sniffer` entre vírgula, ponto e vírgula, tabulação e barra vertical. Para lotes JSON, verificamos UTF-8 e achatamos apenas os nove campos de interesse, evitando normalizar objetos aninhados de vários gigabytes. Primeiro lemos cinco registros; somente depois percorremos todas as fontes. Formatos desconhecidos geram erro explícito.


In [ ]:
sample_preview, preview_format = read_initial_sample(source_files, rows=5)
print(preview_format)
display(sample_preview)
display(pd.DataFrame({"source_column": sample_preview.columns,
                      "normalized": [str(c).lower() for c in sample_preview.columns]}))


{'format': 'json_batches', 'encoding': 'utf-8', 'files': '713'}


,movie_id,title,release_year,rating,votes,genre,runtime_minutes,country,language
0,tt31868189,Happy Gilmore 2,2025,6.3,47513,"Comedy, Sport",114.0,United States,English
1,tt10676052,The Fantastic Four: First Steps,2025,7.5,68405,"Action, Adventure, Sci-Fi",115.0,"United Kingdom, United States, Canada, New Zea...",English
2,tt5950044,Superman,2025,7.6,178182,"Action, Adventure, Sci-Fi",129.0,"United States, Canada, Australia, New Zealand",English
3,tt0116483,Happy Gilmore,1996,7.0,278714,"Comedy, Sport",92.0,United States,English
4,tt28037987,Saiyaara,2025,7.1,38111,"Drama, Romance",150.0,India,Hindi


,source_column,normalized
0,movie_id,movie_id
1,title,title
2,release_year,release_year
3,rating,rating
4,votes,votes
5,genre,genre
6,runtime_minutes,runtime_minutes
7,country,country
8,language,language


### Carregamento e diagnóstico

Agora carregamos os dados sem incluir esse tempo nos benchmarks. Exibimos dimensões, tipos, ausências, duplicidades completas e o mapeamento candidato para ID, título, ano, nota, votos, gênero, duração, país e idioma. Colunas ausentes permanecem ausentes; não são inventadas.


In [6]:
raw_movies, detected_format = read_dataset(source_files)
print(f"Registros: {len(raw_movies):,}; colunas: {raw_movies.shape[1]}")
display(raw_movies.head())
display(raw_movies.dtypes.rename("dtype").to_frame())
display(raw_movies.isna().sum().sort_values(ascending=False).rename("missing").to_frame())
print(f"Linhas completamente duplicadas: {raw_movies.duplicated().sum():,}")
proposed_mapping = infer_column_mapping([str(c) for c in raw_movies.columns])
display(pd.Series(proposed_mapping, name="source_column").rename_axis("canonical_field").to_frame())


Registros: 712,794; colunas: 9


,movie_id,title,release_year,rating,votes,genre,runtime_minutes,country,language
0,tt31868189,Happy Gilmore 2,2025.0,6.3,47513,"Comedy, Sport",114.0,United States,English
1,tt10676052,The Fantastic Four: First Steps,2025.0,7.5,68405,"Action, Adventure, Sci-Fi",115.0,"United Kingdom, United States, Canada, New Zea...",English
2,tt5950044,Superman,2025.0,7.6,178182,"Action, Adventure, Sci-Fi",129.0,"United States, Canada, Australia, New Zealand",English
3,tt0116483,Happy Gilmore,1996.0,7.0,278714,"Comedy, Sport",92.0,United States,English
4,tt28037987,Saiyaara,2025.0,7.1,38111,"Drama, Romance",150.0,India,Hindi


,dtype
movie_id,str
title,str
release_year,float64
rating,object
votes,int64
genre,str
runtime_minutes,float64
country,str
language,str


,missing
rating,384237
runtime_minutes,263218
release_year,106653
language,95953
genre,77133
country,15841
title,176
movie_id,0
votes,0


Linhas completamente duplicadas: 0


,source_column
canonical_field,
movie_id,movie_id
title,title
release_year,release_year
rating,rating
votes,votes
genre,genre
runtime_minutes,runtime_minutes
country,country
language,language


## Limpeza e preparação

Removemos somente registros sem ID válido e IDs repetidos, registrando contagens antes/depois. Campos numéricos aceitam símbolos e são convertidos com coerção explícita; anos fora de 1870–2100 viram ausentes. Textos ausentes continuam `NA`. Se IDs como `tt123` contiverem números únicos, esses números formam `index_key`; caso contrário, uma enumeração da lista de IDs ordenada fornece uma chave inteira determinística. O ID textual original é preservado.

A tabela não é ordenada aqui: sua ordem aleatória original será preservada e uma permutação reprodutível será usada nas amostras.


In [7]:
movies, quality_report, column_mapping = clean_movies(raw_movies)
movies.to_csv(config.processed_data_dir / "movies_clean.csv", index=False)
quality_report.to_csv(config.processed_data_dir / "quality_report.csv", index=False)
with (config.processed_data_dir / "column_mapping.json").open("w", encoding="utf-8") as stream:
    json.dump(column_mapping, stream, ensure_ascii=False, indent=2)
display(quality_report)
display(movies.head())
display(movies.isna().sum().rename("missing_after_cleaning").to_frame())
assert movies.movie_id.is_unique and movies.index_key.is_unique


,step,before,after,removed,missing_values_final
0,remove_missing_key,712794,712794,0,NaN
1,remove_duplicate_key,712794,712794,0,NaN
2,final,712794,712794,0,943211.0


,index_key,movie_id,title,release_year,rating,votes,genre,runtime_minutes,country,language
0,31868189,tt31868189,Happy Gilmore 2,2025,6.3,47513,"Comedy, Sport",114,United States,English
1,10676052,tt10676052,The Fantastic Four: First Steps,2025,7.5,68405,"Action, Adventure, Sci-Fi",115,"United Kingdom, United States, Canada, New Zea...",English
2,5950044,tt5950044,Superman,2025,7.6,178182,"Action, Adventure, Sci-Fi",129,"United States, Canada, Australia, New Zealand",English
3,116483,tt0116483,Happy Gilmore,1996,7.0,278714,"Comedy, Sport",92,United States,English
4,28037987,tt28037987,Saiyaara,2025,7.1,38111,"Drama, Romance",150,India,Hindi


,missing_after_cleaning
index_key,0
movie_id,0
title,176
release_year,106653
rating,384237
votes,0
genre,77133
runtime_minutes,263218
country,15841
language,95953


## Amostras experimentais

Uma única permutação com semente 42 gera amostras aninhadas de 10 mil, 100 mil e 700 mil filmes. Tamanhos maiores que o conjunto limpo são omitidos e registrados. Para cada tamanho, ambas as árvores recebem exatamente os mesmos itens na mesma ordem aleatória reproduzível.


In [8]:
samples = reproducible_samples(
    movies, config.sample_sizes, config.random_seed, config.include_full_dataset
)
sample_manifest = pd.DataFrame({
    "requested_size": config.sample_sizes,
    "executed": [size in samples for size in config.sample_sizes],
})
display(sample_manifest)
print("Escalas executadas:", list(samples))


,requested_size,executed
0,10000,True
1,100000,True
2,700000,True


Escalas executadas: [10000, 100000, 700000]


## Banco SQLite: registros versus índices

Um **registro** é a linha completa do filme. Um **índice** guarda uma chave pequena e uma referência (`index_key`) ao registro. O **nó** é a unidade lógica das árvores; uma **página de banco** é a unidade de armazenamento do SQLite. Cada visita a nó conta como página lógica simulada, não leitura física.

A tabela usa `index_key INTEGER PRIMARY KEY`, ID textual previamente validado como único e tipos apropriados. A carga usa `executemany` dentro de transação. Índices SQLite adicionais só são criados na etapa comparativa. Para tornar o efeito observável, as consultas pontuais de referência usam o `movie_id` textual: primeiro sem índice extra, depois com índice explícito por `movie_id`, e por fim também com `(release_year, index_key)`.


In [9]:
db_path = PROJECT_ROOT / "streaming_catalog.db"
connection = connect_database(db_path)
create_catalog(connection, movies)
print({**database_pages(connection), "file_size_bytes": db_path.stat().st_size})
display(pd.read_sql_query("SELECT * FROM movies LIMIT 5", connection))


{'page_size': 4096, 'page_count': 15713, 'file_size_bytes': 64360448}


,index_key,movie_id,title,release_year,rating,votes,genre,runtime_minutes,country,language
0,9,tt0000009,Miss Jerry,1894,5.4,227,Romance,45,United States,None
1,147,tt0000147,The Corbett-Fitzsimmons Fight,1897,5.3,563,"Documentary, News, Sport",100,United States,None
2,502,tt0000502,Bohemios,1905,3.6,22,NaN,100,Spain,None
3,574,tt0000574,The Story of the Kelly Gang,1906,6.0,1007,"Action, Adventure, Biography, Crime, Drama, Hi...",70,Australia,None
4,591,tt0000591,The Prodigal Son,1907,5.4,33,Drama,90,France,"None, French"


## Estruturas implementadas e instrumentação

Aqui **ordem $m$** significa no máximo $m$ filhos e $m-1$ chaves por nó. Testamos 32, 128 e 256: capacidades pequena, média e grande. Uma ordem compatível com 4 KiB dependeria do tamanho serializado real de chave, ponteiro e cabeçalho; os valores testados representam cenários de páginas progressivamente mais largas, não uma identidade byte a byte.

Na **Árvore B**, a mediana e seu valor são promovidos durante um split; registros podem estar em folhas ou internos. Na **B+**, separadores internos não têm valores: todos os registros ficam nas folhas, e o primeiro item da metade direita vira separador. As folhas têm ponteiros `next_leaf`, permitindo localizar uma vez e percorrer intervalos sequencialmente.

As classes expõem `insert`, `delete`, `search`, `range_search`, `height`, `count_nodes`, `reset_metrics` e `get_metrics`. A remoção implementa empréstimos, fusões e redução da raiz para preservar os limites de ocupação. Contamos comparações, nós, leituras/escritas lógicas e splits. `tracemalloc` mede pico de alocações Python; cada índice é gravado com `pickle` em arquivo temporário real e seu tamanho é medido. O arquivo ainda representa objetos Python, não páginas físicas otimizadas.


In [ ]:
display(pd.DataFrame([
    {"structure": "B-tree", "storage": "RAM/Python", "values": "leaves and internal nodes",
     "linked_leaves": False, "license": "MIT", "order_definition": "max children"},
    {"structure": "B+ tree", "storage": "RAM/Python", "values": "leaves only",
     "linked_leaves": True, "license": "MIT", "order_definition": "max children"},
]))


,structure,storage,values,linked_leaves,license,order_definition
0,B-tree,RAM/Python,leaves and internal nodes,False,MIT,max children
1,B+ tree,RAM/Python,leaves only,True,MIT,max children


## Validação de correção

Desempenho só é medido depois da correção. Os testes cobrem árvore vazia, um item, atualização de duplicata, centenas de chaves em ordem adversa, múltiplos splits, ausência, intervalos inclusivos e ordenação. As saídas são comparadas com um dicionário/lista ordenada; abaixo também conferimos o índice secundário contra pandas. Um `assert` interrompe o notebook em qualquer divergência.


In [ ]:
run_all_validations()
validation_sample = next(iter(samples.values()))
for tree_class in (BTree, BPlusTree):
    primary = tree_class(order=8)
    for row in validation_sample.itertuples():
        primary.insert(int(row.index_key), int(row.index_key))
    assert all(primary.search(int(key)) == int(key) for key in validation_sample.index_key)

    secondary = tree_class(order=8)
    expected_pairs = []
    for row in validation_sample.dropna(subset=["release_year"]).itertuples():
        pair = (int(row.release_year), int(row.index_key))
        secondary.insert(pair, int(row.index_key))
        if 1990 <= pair[0] <= 2000:
            expected_pairs.append((pair, int(row.index_key)))
    assert secondary.range_search((1990, -1), (2000, 10**19)) == sorted(expected_pairs)
print("Todas as validações passaram.")


Todas as validações passaram.


## Metodologia dos benchmarks

Avaliamos índice primário `index_key -> index_key` e índices secundários para categoria e nome. A tupla preserva ordenação e evita perder filmes repetidos. Em cada combinação, medimos 150 buscas por ID, 150 buscas por categoria, 150 buscas por nome, 150 inserções de chaves novas e 150 remoções de chaves existentes. Duas repetições independentes totalizam 300 medições de cada operação.

Antes de rodadas há `gc.collect`; o relógio é `perf_counter_ns`; não há impressão nos laços; cada repetição é salva; a ordem B/B+ é alternada. O CSV e os gráficos ficam fora do trecho medido. Buscas têm aquecimento. Não limpamos o cache do SO: as árvores estão na RAM e “cache frio” não é controlável de forma segura/portátil. Portanto relatamos cache aquecido/estado normal e não alegamos cache físico frio.

Em B+, intervalo registra localização da folha e travessia separadamente. Na B, a travessia em ordem é recursiva e inseparável sem mudar o algoritmo; campos correspondentes ficam ausentes. `perf_counter_ns` e `process_time_ns` fornecem a resolução de medição, mas todos os tempos são convertidos e salvos em **milissegundos**. Eles incluem overhead Python e nenhum equivale a I/O físico.


In [ ]:
experiment_design = pd.DataFrame({
    "parameter": ["mode", "sizes", "orders", "repetitions", "queries", "seed",
                  "insertion_order"],
    "value": [config.run_mode, list(samples), list(config.orders), config.repetitions,
              config.query_count, config.random_seed, "random"],
})
display(experiment_design)


,parameter,value
0,mode,quick
1,sizes,"[10000, 100000, 700000]"
2,orders,"[32, 128, 256]"
3,repetitions,2
4,queries,150
5,seed,42
6,insertion_order,random


## Execução das cinco operações

Executamos busca por ID, categoria e nome, além de inserção e remoção no catálogo, para as nove combinações de ordem e tamanho. A carga inicial é aleatória e reproduzível. Buscas reutilizam o mesmo índice; inserção e remoção recebem uma árvore reconstruída em cada repetição para que todas comecem no mesmo estado. Cada cenário é salvo imediatamente em milissegundos.


In [ ]:
catalog_path = config.raw_results_dir / "catalog_operation_benchmarks.csv"
catalog_results = run_catalog_operation_experiments(samples, config, catalog_path)
print(f"Medições das cinco operações: {len(catalog_results):,}")
display(catalog_results.groupby(["operation", "structure", "order", "sample_size"]).size().rename("measurements").to_frame())


Medições das cinco operações: 27,000


measurements
operation       structure order sample_size              
delete_catalog  B+ tree   32    10000                 300
                                100000                300
                                700000                300
                          128   10000                 300
                                100000                300
...                                                   ...
search_by_title B-tree    128   100000                300
                                700000                300
                          256   10000                 300
                                100000                300
                                700000                300

[90 rows x 1 columns]

### Operações apresentadas nos gráficos

Os gráficos finais usam cinco operações próximas da interface de um catálogo. Categorias são separadas individualmente; nomes e categorias são normalizados sem diferenciar maiúsculas ou acentos. As chaves secundárias são `(categoria, movie_id)` e `(nome, movie_id)`, preservando filmes repetidos. Inserções usam identificadores novos e remoções selecionam identificadores existentes sem reposição.


In [ ]:
display(catalog_results.head())


,structure,index_type,sample_size,order,insertion_order,operation,operation_detail,distribution,interval_size,cache_state,...,height,nodes,internal_nodes,leaves,occupancy,count_results,incremental_size_before_bytes,incremental_size_after_bytes,incremental_file_before_bytes,incremental_file_after_bytes
0,B+ tree,primary_id,10000,32,random,search_by_id,NaN,uniform,NaN,warmed_or_normal,...,3,490,22,468,0.689072,1,NaN,NaN,NaN,NaN
1,B+ tree,primary_id,10000,32,random,search_by_id,NaN,uniform,NaN,warmed_or_normal,...,3,490,22,468,0.689072,1,NaN,NaN,NaN,NaN
2,B+ tree,primary_id,10000,32,random,search_by_id,NaN,uniform,NaN,warmed_or_normal,...,3,490,22,468,0.689072,1,NaN,NaN,NaN,NaN
3,B+ tree,primary_id,10000,32,random,search_by_id,NaN,uniform,NaN,warmed_or_normal,...,3,490,22,468,0.689072,1,NaN,NaN,NaN,NaN
4,B+ tree,primary_id,10000,32,random,search_by_id,NaN,uniform,NaN,warmed_or_normal,...,3,490,22,468,0.689072,1,NaN,NaN,NaN,NaN


## Encerramento do SQLite

O SQLite continua representando o armazenamento dos registros completos, mas não entra nos gráficos solicitados. Fechamos explicitamente a conexão antes da consolidação.


In [ ]:
connection.close()
print("Conexão SQLite fechada.")


Conexão SQLite fechada.


## Consolidação e estatística descritiva

Consolidamos exclusivamente as medições desta execução. Calculamos média, mediana, mínimo, máximo, desvio-padrão, P95 e P99 para cada combinação de estrutura, operação, ordem e tamanho.


In [ ]:
raw_results = catalog_results.copy()
combined_path = config.raw_results_dir / "all_benchmarks.csv"
raw_results.to_csv(combined_path, index=False)
summary = save_processed_results(
    raw_results, config.processed_results_dir / "benchmark_summary.csv"
)
display(summary.head(20))


,structure,index_type,sample_size,order,insertion_order,operation,distribution,interval_size,mean_time_ms,mean_cpu_time_ms,...,mean_pages_read,std_pages_read,mean_pages_written,mean_splits,std_splits,mean_height,std_height,mean_results,speedup_b_over_bplus,difference_percent_bplus_vs_b
0,B+ tree,primary_id,10000,32,random,delete_catalog,uniform,not_applicable,0.008111,0.008423,...,3.0,0.0,1.380000,0.000000,0.000000,3.0,0.0,1.0,0.403226,148.000000
1,B+ tree,primary_id,10000,32,random,insert_catalog,uniform,not_applicable,0.002849,0.003297,...,3.0,0.0,1.180000,0.060000,0.237884,3.0,0.0,1.0,1.180638,-15.300000
2,B+ tree,primary_id,10000,32,random,search_by_id,uniform,not_applicable,0.002634,0.003063,...,3.0,0.0,0.000000,0.000000,0.000000,3.0,0.0,1.0,1.047988,-4.579025
3,B+ tree,primary_id,10000,128,random,delete_catalog,uniform,not_applicable,0.012061,0.012463,...,2.0,0.0,1.020000,0.000000,0.000000,2.0,0.0,1.0,0.215250,364.576074
4,B+ tree,primary_id,10000,128,random,insert_catalog,uniform,not_applicable,0.002399,0.002843,...,2.0,0.0,1.040000,0.013333,0.114889,2.0,0.0,1.0,1.143163,-12.523435
5,B+ tree,primary_id,10000,128,random,search_by_id,uniform,not_applicable,0.002622,0.003003,...,2.0,0.0,0.000000,0.000000,0.000000,2.0,0.0,1.0,0.931291,7.377778
6,B+ tree,primary_id,10000,256,random,delete_catalog,uniform,not_applicable,0.009144,0.009620,...,2.0,0.0,1.060000,0.000000,0.000000,2.0,0.0,1.0,0.281984,254.630671
7,B+ tree,primary_id,10000,256,random,insert_catalog,uniform,not_applicable,0.002217,0.002603,...,2.0,0.0,1.020000,0.006667,0.081513,2.0,0.0,1.0,1.019621,-1.924383
8,B+ tree,primary_id,10000,256,random,search_by_id,uniform,not_applicable,0.002771,0.003163,...,2.0,0.0,0.000000,0.000000,0.000000,2.0,0.0,1.0,0.965053,3.621291
9,B+ tree,primary_id,100000,32,random,delete_catalog,uniform,not_applicable,0.012823,0.013283,...,4.0,0.0,1.193333,0.000000,0.000000,4.0,0.0,1.0,0.428304,133.479101


## Gráficos interativos

Cada operação é apresentada em um mapa de calor com dois painéis: Árvore B e Árvore B+. As colunas são as ordens 32, 128 e 256; as linhas são 10 mil, 100 mil e 700 mil filmes. Cada célula contém somente a média das repetições em milissegundos. Os painéis compartilham a mesma escala; tons mais escuros indicam maior tempo.

A coleção final possui exatamente cinco gráficos: busca por ID, busca por categoria, busca por nome, inserção e remoção.


In [ ]:
figures = create_figures(raw_results, summary)
export_figures(
    figures,
    PROJECT_ROOT / "figures" / "html",
    PROJECT_ROOT / "figures" / "static",
    config.export_static_images,
)
print(f"Figuras geradas: {len(figures)}")
graph_descriptions = [
    ("01_search_by_id", "Busca por ID", "Compara o tempo médio para localizar um único filme pela chave identificadora."),
    ("02_search_by_category", "Busca por categoria", "Compara o tempo médio para obter os filmes ligados a uma categoria."),
    ("03_search_by_title", "Busca por nome", "Compara o tempo médio para localizar filmes com o mesmo título normalizado."),
    ("04_insert_catalog", "Inserção", "Compara o tempo médio para adicionar um novo filme ao índice primário."),
    ("05_delete_catalog", "Remoção", "Compara o tempo médio para remover um filme existente e rebalancear a árvore."),
]
for name, heading, description in graph_descriptions:
    if name in figures:
        display(Markdown(f"### {heading}\n\n{description} Em todos os casos, **menor é melhor**."))
        figures[name].show()


Figuras geradas: 5


### Busca por ID

Compara o tempo médio para localizar um único filme pela chave identificadora. Em todos os casos, **menor é melhor**.

### Busca por categoria

Compara o tempo médio para obter os filmes ligados a uma categoria. Em todos os casos, **menor é melhor**.

### Busca por nome

Compara o tempo médio para localizar filmes com o mesmo título normalizado. Em todos os casos, **menor é melhor**.

### Inserção

Compara o tempo médio para adicionar um novo filme ao índice primário. Em todos os casos, **menor é melhor**.

### Remoção

Compara o tempo médio para remover um filme existente e rebalancear a árvore. Em todos os casos, **menor é melhor**.

### Avaliação cautelosa das hipóteses

O código abaixo usa critérios transparentes e conservadores. “Apoiada” descreve apenas estes dados/configurações; “inconclusiva” é preferível quando faltam pontos ou a diferença agregada é pequena. H1 permanece uma avaliação de tendência empírica, pois complexidade assintótica não é provada por regressão de poucos tamanhos.


In [ ]:
def evaluate_hypotheses(table: pd.DataFrame) -> pd.DataFrame:
    outcomes = []
    trees = table[table.structure.isin(["B-tree", "B+ tree"])]
    point = trees[trees.operation == "search_by_id"]
    outcomes.append(("H1", "apoiada com ressalvas" if point.sample_size.nunique() == 3 else "inconclusiva",
                     "A busca por ID foi medida em três escalas; isso não substitui a prova O(log n)."))
    category = trees[trees.operation == "search_by_category"]
    pivot = category.groupby("structure")["median_time_ms"].median()
    if {"B-tree", "B+ tree"}.issubset(pivot.index):
        ratio = pivot["B-tree"] / pivot["B+ tree"]
        status = "apoiada" if ratio > 1.05 else "rejeitada" if ratio < 0.95 else "inconclusiva"
        reason = f"Razão agregada B/B+ na busca por categoria: {ratio:.3f}."
    else:
        status, reason = "inconclusiva", "Medições insuficientes."
    outcomes.append(("H2", status, reason))
    heights = trees.groupby("order")["mean_height"].median().sort_index()
    h3 = "apoiada" if len(heights) >= 2 and heights.iloc[-1] <= heights.iloc[0] else "inconclusiva"
    outcomes.append(("H3", h3, f"Alturas medianas por ordem: {heights.to_dict()}."))
    outcomes.append(("H4", "inconclusiva", "A execução atual fixa inserção aleatória e não compara ordem de entrada."))
    outcomes.append(("H5", "inconclusiva", "Exigiria controlar outras linguagens e modelos de armazenamento."))
    outcomes.append(("H6", "apoiada em alguns cenários" if not point.empty else "inconclusiva",
                     "Consulte as nove células de busca por ID; a agregação não garante vantagem geral."))
    return pd.DataFrame(outcomes, columns=["hypothesis", "assessment", "evidence"])

hypothesis_results = evaluate_hypotheses(summary)
display(hypothesis_results)


,hypothesis,assessment,evidence
0,H1,apoiada com ressalvas,A busca por ID foi medida em três escalas; iss...
1,H2,apoiada,Razão agregada B/B+ na busca por categoria: 1....
2,H3,apoiada,"Alturas medianas por ordem: {32: 4.0, 128: 3.0..."
3,H4,inconclusiva,A execução atual fixa inserção aleatória e não...
4,H5,inconclusiva,Exigiria controlar outras linguagens e modelos...
5,H6,apoiada em alguns cenários,Consulte as nove células de busca por ID; a ag...


## Análise técnica

Ambas as árvores limitam a altura por um fator de ramificação e têm busca $O(\log n)$. Isso não implica tempos iguais: a B pode terminar num nó interno; a B+ sempre alcança folha. Em contrapartida, a B+ concentra valores nas folhas, aumenta fan-out interno e, após localizar o início, percorre `next_leaf` com boa localidade lógica. A B precisa realizar travessia em ordem por subárvores.

Ordem maior tende a reduzir altura, mas sua busca binária interna ainda faz comparações e nós grandes ampliam trabalho de split, alocações e espaço ocioso. Inserção ordenada pode repetir splits na borda direita e produzir ocupação distinta. Python acrescenta listas dinâmicas, objetos inteiros, coletor de lixo e interpretador; por isso o tempo real reflete constantes e localidade além da notação assintótica.

“Página lida” neste estudo significa chamada de visita a um nó. “Página escrita” significa nó logicamente alterado. É uma aproximação algorítmica controlável, não uma chamada ao disco. SQLite possui cache de páginas verdadeiro e seu arquivo inclui tabela, índices e metadados. O arquivo temporário `pickle` mede representação serializada das árvores, não um layout otimizado de 4 KiB.


## Ameaças à validade

- **Implementação:** Python acadêmico não representa bancos em C/Rust; escolhas de lista e busca binária afetam constantes.
- **Ambiente:** SO, escalonamento, processos de fundo, frequência da CPU, GC e cache interferem. Não limpamos cache físico.
- **Memória:** `tracemalloc` observa alocações Python, não necessariamente todo RSS; serialização não equivale a página de índice.
- **Páginas:** os contadores são lógicos; nenhuma conclusão deve chamá-los de I/O físico.
- **Carga:** o dataset é real, mas as consultas por ID, categoria e nome são amostradas uniformemente e não cobrem todo padrão de catálogo.
- **Amostragem:** duas repetições e 300 medições por operação podem ser insuficientes para ruído raro; operações da mesma execução não são totalmente independentes.
- **SQLite:** armazena os registros completos, mas não participa dos cinco gráficos atuais.
- **Generalização:** resultados valem para este hardware, versão, dados, ordens e implementação. Ausências de colunas no dataset podem reduzir o índice por ano.


## Conclusão

A conclusão quantitativa deve nascer da execução. A próxima célula resume vencedores observados e estado das hipóteses; não extrapola para todo hardware. Conceitualmente, o experimento separa o que a teoria garante — altura logarítmica — daquilo que precisa ser medido: constantes, splits, ocupação, localidade, custo Python, seletividade e páginas lógicas.


In [ ]:
conclusion_lines = [numeric_interpretation(summary, operation) for operation in ("search_by_id", "search_by_category", "search_by_title", "insert_catalog", "delete_catalog")]
conclusion_lines.append("As conclusões são condicionais ao ambiente registrado e aos intervalos de dispersão exibidos.")
display(Markdown("\n\n".join(conclusion_lines)))


Em search_by_id, B+ tree apresentou a menor mediana agregada (0.003291 ms), uma diferença de 7.1% frente à outra estrutura. A mediana dos desvios-padrão por estrutura variou de 0.000789 a 0.001314 ms. A diferença deve ser interpretada junto das dispersões e configurações individuais.

Em search_by_category, B+ tree apresentou a menor mediana agregada (0.649145 ms), uma diferença de 30.4% frente à outra estrutura. A mediana dos desvios-padrão por estrutura variou de 0.209173 a 0.268414 ms. A diferença deve ser interpretada junto das dispersões e configurações individuais.

Em search_by_title, B+ tree apresentou a menor mediana agregada (0.004583 ms), uma diferença de 12.3% frente à outra estrutura. A mediana dos desvios-padrão por estrutura variou de 0.000531 a 0.000589 ms. A diferença deve ser interpretada junto das dispersões e configurações individuais.

Em insert_catalog, B-tree apresentou a menor mediana agregada (0.003000 ms), uma diferença de 1.3% frente à outra estrutura. A mediana dos desvios-padrão por estrutura variou de 0.000862 a 0.001309 ms. A diferença deve ser interpretada junto das dispersões e configurações individuais.

Em delete_catalog, B-tree apresentou a menor mediana agregada (0.004209 ms), uma diferença de 65.9% frente à outra estrutura. A mediana dos desvios-padrão por estrutura variou de 0.001844 a 0.003689 ms. A diferença deve ser interpretada junto das dispersões e configurações individuais.

As conclusões são condicionais ao ambiente registrado e aos intervalos de dispersão exibidos.